# Train and test a probe on judge-labeled activations

Trains a linear probe on the natural-generation activations from `data-<dataset>/<model>.pt`,
with labels derived from a `scored-*-<model>.json` file (filtered by `judge_model`).

**Default probe**: regularized linear logistic regression (StandardScaler + sklearn
`LogisticRegression(C=1/reg_coeff, fit_intercept=False)`), mirroring the `lr` method
in `deception_detection/scripts/configs/repe_llama-8b_layer22.yaml` (`reg_coeff: 10`).

**Categories**: judge scores in `[-10, 10]` are bucketed into `{-2, -1, 0, 1, 2}` via
the standard split (`<-5`, `<0`, `==0`, `<=5`, `>5`). A user-supplied
`CATEGORY_TO_LABEL` dict maps categories to class labels; categories absent
from the dict are dropped. The default is binary `{-2: -1, 2: 1}` — multi-class
is supported by including more entries.

**Train/test split**: deterministic 90/10 over **conversations** (not tokens), so all
tokens from a conversation go to a single split.

## 1 · Configuration

In [17]:
# What to train on ────────────────────────────────────────────────────────
MODEL    = "meta-llama/Llama-3.1-8B-Instruct"   # probe model (also subject)
LAYER    = 16                                    # absolute transformer layer
DATASETS = "more-roleplay"                       # str or list of str ("data-" prefix optional)
JUDGE    = "gpt-5.4-nano"                        # value of `judge_model` in scored-*.json

# Token-position window within each generation (inclusive, 0-indexed).
# Generations are up to m tokens long (m=100 in current datasets); padding
# positions are dropped automatically. JMIN=0, JMAX=m-1 uses every position.
JMIN = 0
JMAX = 10

# Category → label mapping. Categories not listed are dropped.
#   -2 = score < -5    (strongly deceptive)
#   -1 = -5 <= score < 0
#    0 = score == 0
#    1 = 0 < score <= 5
#    2 = score > 5     (strongly honest)
CATEGORY_TO_LABEL = {-2: -1, 2: 1}                # default binary
# Example multi-class: {-2: 0, -1: 1, 0: 2, 1: 3, 2: 4}

# Probe hyperparameters (mirrors repe_llama-8b_layer22.yaml)
PROBE_KIND   = "lr"          # "lr" = per-token logistic regression (default)
                              # "mlr" = mean-over-valid-tokens logistic regression
REG_COEFF    = 1.0           # sklearn C = 1 / reg_coeff
NORMALIZE    = True           # StandardScaler before fitting
FIT_INTERCEPT = False         # matches LogisticRegressionDetector

# Split
TEST_FRACTION = 0.10
RANDOM_SEED   = 42

# Path root (autocomp2 dir). Adjust if running elsewhere.
ROOT = "."

## 2 · Imports & helpers

In [18]:
import json
import os
import re
from collections import Counter

import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SCORED_RE = re.compile(
    r"^scored-(?:(?P<variant>[a-zA-Z0-9]+)-)?\d{8}-\d{6}-(?P<model>.+)\.json$"
)


def model_to_filename(model_id: str) -> str:
    """`meta-llama/Llama-3.1-8B-Instruct` -> `meta-llama--Llama-3.1-8B-Instruct`."""
    return model_id.replace("/", "--")


def categorize(score):
    """Bucket a -10..10 judge score into {-2, -1, 0, 1, 2}."""
    if score < -5:
        return -2
    if score < 0:
        return -1
    if score == 0:
        return 0
    if score <= 5:
        return 1
    return 2


def normalize_dataset_arg(arg):
    """Coerce DATASETS to a list of `data-<name>` directory paths under ROOT."""
    if isinstance(arg, str):
        arg = [arg]
    out = []
    for d in arg:
        d = d if d.startswith("data-") else f"data-{d}"
        out.append(os.path.join(ROOT, d))
    return out

## 3 · Load activations and labels

For each dataset directory:
- read the model's `.json` to find the positional index of `LAYER` in its `layers` list,
- read the `.pt` and slice `autocompletion_activations[:, 0, :, layer_idx, :]` (natural i=0 generation),
- find the latest `scored-*-<model>.json` with `judge_model == JUDGE`,
- bucket scores into categories and map through `CATEGORY_TO_LABEL`.

Padding tokens (rows that are all zero in the activation tensor) are filtered out.

In [19]:
def find_scored_for_judge(data_dir: str, model_fname: str, judge: str,
                          exclude_variants=("ac1", "ac3", "ac5")) -> dict:
    """Return `{index: score}` from the latest scored-*.json matching judge."""
    chosen_path = None
    for f in sorted(os.listdir(data_dir)):                # sorted -> latest wins
        m = SCORED_RE.match(f)
        if not m or m.group("model") != model_fname:
            continue
        if m.group("variant") in exclude_variants:
            continue
        with open(os.path.join(data_dir, f)) as fh:
            d = json.load(fh)
        if d.get("judge_model") == judge:
            chosen_path = os.path.join(data_dir, f)
            chosen = d
    if chosen_path is None:
        raise FileNotFoundError(
            f"No scored-*-{model_fname}.json with judge_model={judge!r} in {data_dir}")
    print(f"  scored file: {os.path.basename(chosen_path)}  (n={len(chosen['scores'])})")
    return {s["index"]: s["score"] for s in chosen["scores"] if "score" in s}


def load_split(data_dir: str, model_id: str, layer: int, judge: str,
               cat_to_label: dict, jmin: int = 0, jmax: int | None = None):
    """Load (acts, labels, conv_ids, valid_mask) for one dataset.

    Only token positions j with `jmin <= j <= jmax` (inclusive, 0-indexed) are
    kept. `jmax=None` means "to the end of the generation".

    acts:        (N*M, d) float32  — natural-generation token activations,
                                      restricted to the [jmin, jmax] window
    labels:      (N*M,) int        — class labels (per token, inherited from conv)
    conv_ids:    (N*M,) int        — conversation k (for split + aggregation)
    valid_mask:  (N*M,) bool       — drops padding (all-zero rows) and conversations
                                      whose category isn't in cat_to_label.
    """
    fname = model_to_filename(model_id)
    json_path = os.path.join(data_dir, f"{fname}.json")
    pt_path = os.path.join(data_dir, f"{fname}.pt")

    with open(json_path) as f:
        meta = json.load(f)
    if layer not in meta["layers"]:
        raise ValueError(
            f"layer={layer} not in {meta['layers']} for {json_path}")
    layer_idx = meta["layers"].index(layer)

    print(f"Loading {data_dir}/{fname}  layer={layer} (idx={layer_idx})")
    pt = torch.load(pt_path, map_location="cpu", weights_only=True)
    ac_tensor = pt["autocompletion_activations"]          # (N, c, m, L, d)
    N, c, m, L, d = ac_tensor.shape

    # Token-position window
    if jmax is None or jmax >= m:
        jmax = m - 1
    if jmin < 0 or jmin > jmax:
        raise ValueError(f"Invalid window: jmin={jmin}, jmax={jmax}, m={m}")
    M = jmax - jmin + 1
    print(f"  token window: j in [{jmin}, {jmax}]  ({M} positions out of m={m})")

    slab = ac_tensor[:, 0, jmin:jmax + 1, layer_idx, :].float().numpy()  # (N, M, d) — i=0 natural

    idx_to_score = find_scored_for_judge(data_dir, fname, judge)

    conv_ks = np.array([conv["k"] for conv in meta["conversations"]])  # (N,)
    if len(conv_ks) != N:
        raise ValueError(f"N mismatch: pt={N} vs json={len(conv_ks)} in {data_dir}")

    # Per-conversation label (or NaN if dropped)
    label_per_conv = np.full(N, np.nan)
    for i, k in enumerate(conv_ks):
        if k not in idx_to_score:
            continue
        cat = categorize(idx_to_score[k])
        if cat in cat_to_label:
            label_per_conv[i] = cat_to_label[cat]

    # Expand to token level
    acts      = slab.reshape(N * M, d)
    labels    = np.repeat(label_per_conv, M)              # (N*M,)
    conv_id   = np.repeat(conv_ks, M)                     # (N*M,)
    is_pad    = ~np.any(acts != 0, axis=1)                # all-zero -> padding
    valid     = (~is_pad) & ~np.isnan(labels)

    print(f"  N={N}  M={M}  d={d}  valid_tokens={valid.sum():,} / {len(valid):,}"
          f"  convs_kept={len(set(conv_id[valid])):,} / {N:,}")
    return acts, labels, conv_id, valid

In [20]:
data_dirs = normalize_dataset_arg(DATASETS)

acts_parts, labels_parts, conv_id_parts, valid_parts = [], [], [], []
_conv_offset = 0  # make conv ids unique across datasets
for dd in data_dirs:
    a, y, k, v = load_split(dd, MODEL, LAYER, JUDGE, CATEGORY_TO_LABEL,
                            jmin=JMIN, jmax=JMAX)
    k = k.astype(np.int64) + _conv_offset
    _conv_offset = int(k.max()) + 1
    acts_parts.append(a)
    labels_parts.append(y)
    conv_id_parts.append(k)
    valid_parts.append(v)

acts      = np.concatenate(acts_parts, axis=0)
labels    = np.concatenate(labels_parts, axis=0)
conv_id   = np.concatenate(conv_id_parts, axis=0)
valid     = np.concatenate(valid_parts, axis=0)

acts   = acts[valid]
labels = labels[valid].astype(np.int64)
conv_id = conv_id[valid]

print()
print(f"Final pooled: tokens={len(labels):,}  unique_convs={len(set(conv_id)):,}")
print("Label distribution (tokens):", dict(sorted(Counter(labels.tolist()).items())))
_per_conv_label = {c: labels[conv_id == c][0] for c in np.unique(conv_id)}
print("Label distribution (convs): ", dict(sorted(Counter(_per_conv_label.values()).items())))

Loading ./data-more-roleplay/meta-llama--Llama-3.1-8B-Instruct  layer=16 (idx=2)
  token window: j in [0, 10]  (11 positions out of m=100)
  scored file: scored-20260513-022705-meta-llama--Llama-3.1-8B-Instruct.json  (n=1000)
  N=1000  M=11  d=4096  valid_tokens=10,009 / 11,000  convs_kept=911 / 1,000

Final pooled: tokens=10,009  unique_convs=911
Label distribution (tokens): {-1: 6567, 1: 3442}
Label distribution (convs):  {-1: 597, 1: 314}


## 4 · Deterministic 90/10 train/test split (by conversation)

Splitting at the conversation level ensures no token-leakage between train and test.

In [21]:
unique_convs = np.array(sorted(set(conv_id.tolist())))
conv_labels  = np.array([_per_conv_label[c] for c in unique_convs])

# Stratify if every class has >=2 convs; otherwise plain split.
_class_counts = Counter(conv_labels.tolist())
stratify = conv_labels if all(n >= 2 for n in _class_counts.values()) else None

train_convs, test_convs = train_test_split(
    unique_convs,
    test_size=TEST_FRACTION,
    random_state=RANDOM_SEED,
    shuffle=True,
    stratify=stratify,
)
train_convs = set(train_convs.tolist())
test_convs  = set(test_convs.tolist())

is_train = np.array([c in train_convs for c in conv_id])
is_test  = ~is_train

print(f"convs:  train={len(train_convs)}  test={len(test_convs)}")
print(f"tokens: train={is_train.sum():,}  test={is_test.sum():,}")

convs:  train=819  test=92
tokens: train=9,001  test=1,008


## 5 · Train the probe

Defaults to per-token LR (`PROBE_KIND="lr"`), matching the repe yaml. Set
`PROBE_KIND="mlr"` to instead train on the mean activation per conversation.

In [22]:
if PROBE_KIND == "lr":
    X_train, y_train = acts[is_train], labels[is_train]
    X_test,  y_test  = acts[is_test],  labels[is_test]
elif PROBE_KIND == "mlr":
    # Mean over the valid tokens of each conversation.
    def _mean_per_conv(mask):
        cids = conv_id[mask]
        order = np.argsort(cids, kind="stable")
        cids_s = cids[order]
        feats_s = acts[mask][order]
        labs_s  = labels[mask][order]
        uniq, starts = np.unique(cids_s, return_index=True)
        ends = np.r_[starts[1:], len(cids_s)]
        X = np.stack([feats_s[s:e].mean(axis=0) for s, e in zip(starts, ends)])
        y = np.array([labs_s[s] for s in starts])
        return X, y, uniq
    X_train, y_train, _ = _mean_per_conv(is_train)
    X_test,  y_test, test_conv_order = _mean_per_conv(is_test)
else:
    raise ValueError(f"Unknown PROBE_KIND={PROBE_KIND!r}")

print(f"X_train={X_train.shape}  X_test={X_test.shape}")
print(f"y_train dist: {dict(sorted(Counter(y_train.tolist()).items()))}")
print(f"y_test  dist: {dict(sorted(Counter(y_test.tolist()).items()))}")

scaler = StandardScaler() if NORMALIZE else None
Xtr = scaler.fit_transform(X_train) if scaler is not None else X_train
Xte = scaler.transform(X_test)      if scaler is not None else X_test

clf = LogisticRegression(
    C=1.0 / REG_COEFF,
    fit_intercept=FIT_INTERCEPT,
    random_state=RANDOM_SEED,
    max_iter=2000,
    solver="lbfgs",
)
clf.fit(Xtr, y_train)
print(f"Fitted LR: classes={clf.classes_.tolist()}  coef={clf.coef_.shape}  C={clf.C}")

X_train=(9001, 4096)  X_test=(1008, 4096)
y_train dist: {-1: 5907, 1: 3094}
y_test  dist: {-1: 660, 1: 348}
Fitted LR: classes=[-1, 1]  coef=(1, 4096)  C=1.0


## 6 · Evaluate

In [23]:
def report(name, X, y, model):
    pred = model.predict(X)
    acc  = accuracy_score(y, pred)
    bacc = balanced_accuracy_score(y, pred)
    print(f"--- {name} ---")
    print(f"  accuracy         = {acc:.4f}")
    print(f"  balanced accuracy = {bacc:.4f}")
    if len(model.classes_) == 2:
        proba = model.predict_proba(X)[:, 1]
        try:
            print(f"  AUC              = {roc_auc_score(y, proba):.4f}")
        except ValueError as e:
            print(f"  AUC              = (skipped: {e})")
    print("  confusion matrix (rows = true, cols = pred):")
    cm = confusion_matrix(y, pred, labels=model.classes_)
    header = "    " + "  ".join(f"p={c:>3}" for c in model.classes_)
    print(header)
    for c, row in zip(model.classes_, cm):
        print(f"    t={c:>3}  " + "  ".join(f"{v:>5d}" for v in row))
    print(classification_report(y, pred, labels=model.classes_, zero_division=0))

report("train", Xtr, y_train, clf)
report("test",  Xte, y_test,  clf)

--- train ---
  accuracy         = 1.0000
  balanced accuracy = 1.0000
  AUC              = 1.0000
  confusion matrix (rows = true, cols = pred):
    p= -1  p=  1
    t= -1   5907      0
    t=  1      0   3094
              precision    recall  f1-score   support

          -1       1.00      1.00      1.00      5907
           1       1.00      1.00      1.00      3094

    accuracy                           1.00      9001
   macro avg       1.00      1.00      1.00      9001
weighted avg       1.00      1.00      1.00      9001

--- test ---
  accuracy         = 0.6726
  balanced accuracy = 0.6563
  AUC              = 0.7260
  confusion matrix (rows = true, cols = pred):
    p= -1  p=  1
    t= -1    468    192
    t=  1    138    210
              precision    recall  f1-score   support

          -1       0.77      0.71      0.74       660
           1       0.52      0.60      0.56       348

    accuracy                           0.67      1008
   macro avg       0.65      0.66 

## 7 · Per-conversation aggregation on the test set

When training per-token (`PROBE_KIND="lr"`), the conversation-level prediction
is the majority vote / mean-probability across that conversation's valid tokens.
Skipped when `PROBE_KIND="mlr"` (already conversation-level).

In [24]:
if PROBE_KIND == "lr":
    test_conv_ids = conv_id[is_test]
    proba_test = clf.predict_proba(Xte)             # (T, K)
    classes = clf.classes_

    by_conv_proba = {}
    by_conv_label = {}
    for c in np.unique(test_conv_ids):
        mask = test_conv_ids == c
        by_conv_proba[c] = proba_test[mask].mean(axis=0)
        by_conv_label[c] = int(y_test[mask][0])

    convs_sorted = np.array(sorted(by_conv_proba))
    P = np.stack([by_conv_proba[c] for c in convs_sorted])
    Y = np.array([by_conv_label[c] for c in convs_sorted])
    pred_conv = classes[P.argmax(axis=1)]

    acc  = accuracy_score(Y, pred_conv)
    bacc = balanced_accuracy_score(Y, pred_conv)
    print(f"Per-conversation accuracy          = {acc:.4f}  (n={len(Y)})")
    print(f"Per-conversation balanced accuracy = {bacc:.4f}")
    if len(classes) == 2:
        try:
            print(f"Per-conversation AUC               = {roc_auc_score(Y, P[:, 1]):.4f}")
        except ValueError as e:
            print(f"AUC skipped: {e}")
    cm = confusion_matrix(Y, pred_conv, labels=classes)
    print("Conversation-level confusion matrix:")
    print("    " + "  ".join(f"p={c:>3}" for c in classes))
    for c, row in zip(classes, cm):
        print(f"    t={c:>3}  " + "  ".join(f"{v:>5d}" for v in row))
else:
    print("(PROBE_KIND='mlr' is already at conversation level — skipping.)")

Per-conversation accuracy          = 0.7174  (n=92)
Per-conversation balanced accuracy = 0.6885
Per-conversation AUC               = 0.7578
Conversation-level confusion matrix:
    p= -1  p=  1
    t= -1     47     13
    t=  1     13     19


## 8 · (Optional) save the probe

Saves directions, scaler params, and config — compatible-in-spirit with
`deception_detection.detectors.LogisticRegressionDetector`.

In [25]:
import pickle

SAVE_PATH = None    # e.g. "probes/llama8b_layer12_lr.pkl" — set to enable

if SAVE_PATH:
    os.makedirs(os.path.dirname(SAVE_PATH) or ".", exist_ok=True)
    payload = {
        "model": MODEL,
        "layer": LAYER,
        "datasets": data_dirs,
        "judge": JUDGE,
        "category_to_label": CATEGORY_TO_LABEL,
        "probe_kind": PROBE_KIND,
        "reg_coeff": REG_COEFF,
        "normalize": NORMALIZE,
        "fit_intercept": FIT_INTERCEPT,
        "classes": clf.classes_.tolist(),
        "coef": clf.coef_,
        "intercept": clf.intercept_,
        "scaler_mean":  None if scaler is None else scaler.mean_,
        "scaler_scale": None if scaler is None else scaler.scale_,
    }
    with open(SAVE_PATH, "wb") as f:
        pickle.dump(payload, f)
    print(f"Saved -> {SAVE_PATH}")
else:
    print("SAVE_PATH is None — not saving.")

SAVE_PATH is None — not saving.


## 9 · Transfer-evaluate on other datasets

Apply the *already-trained* probe (same `MODEL` and `LAYER` are required) to any
other dataset(s). Each kwarg defaults to the training-time value, so the
common case is just `evaluate_on("autoconv10")`. You can override `judge`,
`cat_to_label`, or the token window (`jmin`/`jmax`) independently.

In [26]:
def evaluate_on(
    datasets,
    *,
    model=None, layer=None, judge=None,
    cat_to_label=None, jmin=None, jmax=None,
    aggregate=True, verbose=True,
):
    """Run the trained probe on `datasets` and report metrics.

    Defaults to the training-time settings; override any kwarg as needed.
    `aggregate=True` also reports per-conversation metrics (mean predict_proba
    over each conversation's valid tokens).

    Returns a dict with token-level and (optional) conv-level metrics, plus the
    raw predictions and probabilities for downstream analysis.
    """
    model        = MODEL              if model is None else model
    layer        = LAYER               if layer is None else layer
    judge        = JUDGE               if judge is None else judge
    cat_to_label = CATEGORY_TO_LABEL   if cat_to_label is None else cat_to_label
    jmin         = JMIN                if jmin is None else jmin
    jmax         = JMAX                if jmax is None else jmax

    data_dirs = normalize_dataset_arg(datasets)

    a_parts, y_parts, k_parts, v_parts = [], [], [], []
    offset = 0
    for dd in data_dirs:
        a, y, k, v = load_split(dd, model, layer, judge, cat_to_label,
                                jmin=jmin, jmax=jmax)
        k = k.astype(np.int64) + offset
        offset = int(k.max()) + 1
        a_parts.append(a); y_parts.append(y); k_parts.append(k); v_parts.append(v)
    A = np.concatenate(a_parts, 0)
    Y = np.concatenate(y_parts, 0)
    K = np.concatenate(k_parts, 0)
    V = np.concatenate(v_parts, 0)

    A = A[V]; Y = Y[V].astype(np.int64); K = K[V]
    if len(Y) == 0:
        print("(no valid tokens — nothing to evaluate)")
        return None

    X = scaler.transform(A) if scaler is not None else A

    # If PROBE_KIND='mlr', take the per-conv mean before predicting (matches train).
    if PROBE_KIND == "mlr":
        order = np.argsort(K, kind="stable")
        K_s = K[order]; X_s = X[order]; Y_s = Y[order]
        uniq, starts = np.unique(K_s, return_index=True)
        ends = np.r_[starts[1:], len(K_s)]
        X = np.stack([X_s[s:e].mean(0) for s, e in zip(starts, ends)])
        Y = np.array([Y_s[s] for s in starts])
        K = uniq

    pred  = clf.predict(X)
    proba = clf.predict_proba(X)
    classes = clf.classes_

    if verbose:
        unit = "convs" if PROBE_KIND == "mlr" else "tokens"
        print(f"--- {datasets}  ({len(Y):,} {unit}) ---")
        print(f"  accuracy         = {accuracy_score(Y, pred):.4f}")
        print(f"  balanced accuracy = {balanced_accuracy_score(Y, pred):.4f}")
        if len(classes) == 2:
            try:
                print(f"  AUC              = {roc_auc_score(Y, proba[:, 1]):.4f}")
            except ValueError as e:
                print(f"  AUC              = (skipped: {e})")
        cm = confusion_matrix(Y, pred, labels=classes)
        print("    " + "  ".join(f"p={c:>3}" for c in classes))
        for c, row in zip(classes, cm):
            print(f"    t={c:>3}  " + "  ".join(f"{v:>5d}" for v in row))

    out = {
        "datasets": data_dirs, "judge": judge, "n": len(Y),
        "accuracy": accuracy_score(Y, pred),
        "balanced_accuracy": balanced_accuracy_score(Y, pred),
        "confusion_matrix": confusion_matrix(Y, pred, labels=classes),
        "classes": classes,
        "y_true": Y, "y_pred": pred, "proba": proba, "conv_id": K,
    }
    if len(classes) == 2:
        try:
            out["auc"] = roc_auc_score(Y, proba[:, 1])
        except ValueError:
            out["auc"] = None

    if aggregate and PROBE_KIND == "lr":
        uniq = np.unique(K)
        P, Yc = [], []
        for c in uniq:
            mask = K == c
            P.append(proba[mask].mean(0))
            Yc.append(int(Y[mask][0]))
        P = np.stack(P); Yc = np.array(Yc)
        pred_c = classes[P.argmax(1)]
        out["conv"] = {
            "n": len(Yc), "y_true": Yc, "y_pred": pred_c, "proba": P,
            "accuracy": accuracy_score(Yc, pred_c),
            "balanced_accuracy": balanced_accuracy_score(Yc, pred_c),
            "confusion_matrix": confusion_matrix(Yc, pred_c, labels=classes),
        }
        if len(classes) == 2:
            try:
                out["conv"]["auc"] = roc_auc_score(Yc, P[:, 1])
            except ValueError:
                out["conv"]["auc"] = None
        if verbose:
            cv = out["conv"]
            print(f"  conv-level:  acc={cv['accuracy']:.4f}  "
                  f"bal_acc={cv['balanced_accuracy']:.4f}"
                  + (f"  AUC={cv['auc']:.4f}" if cv.get('auc') is not None else "")
                  + f"  (n={cv['n']})")
    return out


## 10 · Sweep: train many probes and evaluate them all

The cells above use module-level globals (handy for interactive work). For
sweeps, the helpers below take all settings as explicit args, so each probe
gets a fresh scaler / classifier and there's no cross-contamination.

**Caching.** Every probe is keyed by a 12-char hash of its canonical config and
saved under `autocomp2/probes/<hash>/`:
- `config.json` — human-readable cfg (also the hash and `data_dirs`).
- `probe.pkl`   — scaler + classifier + holdout metrics + train/test split.
- `results.pkl` — `{eval_key: metrics}` for any transfer evaluations.

Re-running `train_probe` / `evaluate_probe` / `run_experiments` with the same
config hits the cache (no retraining, no re-loading the .pt). Pass
`force_retrain=True` to recompute the probe; `force_reeval=True` to recompute
a transfer evaluation.

- `train_probe(**cfg)` — load + split + fit (or load from cache); returns the
  `probe` dict.
- `evaluate_probe(probe, datasets, **overrides)` — apply an already-trained
  probe to any other dataset(s) (or load cached result).
- `run_experiments(train_cfgs, eval_on=...)` — train each config; evaluate every
  probe on a shared set of eval datasets.
- `grid_configs(**axes)` — expand a small grid of axes into a list of configs.
- `summarize_results(results)` — print the hyperparameter+metric table.

In [27]:
import hashlib
import itertools
import pickle
from sklearn.metrics import roc_curve


# ── Defaults: every kwarg falls back to the top-of-notebook config ──────────
_DEFAULTS = dict(
    model=MODEL, layer=LAYER, datasets=DATASETS, judge=JUDGE,
    cat_to_label=CATEGORY_TO_LABEL, jmin=JMIN, jmax=JMAX,
    probe_kind=PROBE_KIND, reg_coeff=REG_COEFF,
    normalize=NORMALIZE, fit_intercept=FIT_INTERCEPT,
    test_fraction=TEST_FRACTION, random_seed=RANDOM_SEED,
)
_VALID_KEYS = set(_DEFAULTS) | {"verbose"}

# Cache location: autocomp2/probes/<cfg_hash>/{config.json, probe.pkl, results.pkl}
CACHE_DIR = os.path.join(ROOT, "probes")


# ── Hashing & cache paths ───────────────────────────────────────────────────

def _canon_cfg(cfg):
    """JSON-safe canonical form for hashing/saving — sorted keys, deterministic."""
    c = dict(cfg)
    c.pop("verbose", None)
    c["cat_to_label"] = sorted([list(p) for p in c["cat_to_label"].items()])
    if not isinstance(c["datasets"], str):
        c["datasets"] = list(c["datasets"])
    return c

def _hash(obj) -> str:
    s = json.dumps(obj, sort_keys=True, default=str)
    return hashlib.sha1(s.encode()).hexdigest()[:12]

def _cfg_hash(cfg)  -> str: return _hash(_canon_cfg(cfg))

def _eval_canon(eval_cfg):
    return {
        "datasets":     eval_cfg["datasets"] if isinstance(eval_cfg["datasets"], str)
                        else list(eval_cfg["datasets"]),
        "judge":        eval_cfg["judge"],
        "cat_to_label": sorted([list(p) for p in eval_cfg["cat_to_label"].items()]),
        "jmin":         eval_cfg["jmin"],
        "jmax":         eval_cfg["jmax"],
    }

def _eval_key(eval_cfg) -> str: return _hash(_eval_canon(eval_cfg))

def _probe_dir(cfg_hash, cache_dir=None):
    return os.path.join(cache_dir or CACHE_DIR, cfg_hash)

def _paths(cfg_hash, cache_dir=None):
    d = _probe_dir(cfg_hash, cache_dir)
    return {
        "dir":     d,
        "config":  os.path.join(d, "config.json"),
        "probe":   os.path.join(d, "probe.pkl"),
        "results": os.path.join(d, "results.pkl"),
    }

def _load_results(cfg_hash, cache_dir=None) -> dict:
    p = _paths(cfg_hash, cache_dir)["results"]
    if not os.path.exists(p):
        return {}
    with open(p, "rb") as f:
        return pickle.load(f)

def _save_results(cfg_hash, results, cache_dir=None):
    paths = _paths(cfg_hash, cache_dir)
    os.makedirs(paths["dir"], exist_ok=True)
    with open(paths["results"], "wb") as f:
        pickle.dump(results, f)


# ── Loading (unchanged) ─────────────────────────────────────────────────────

def _load_pooled(model, layer, datasets, judge, cat_to_label, jmin, jmax,
                 verbose=True):
    data_dirs = normalize_dataset_arg(datasets)
    a_parts, y_parts, k_parts, v_parts = [], [], [], []
    offset = 0
    for dd in data_dirs:
        if not verbose:
            import io, contextlib
            with contextlib.redirect_stdout(io.StringIO()):
                a, y, k, v = load_split(dd, model, layer, judge, cat_to_label,
                                        jmin=jmin, jmax=jmax)
        else:
            a, y, k, v = load_split(dd, model, layer, judge, cat_to_label,
                                    jmin=jmin, jmax=jmax)
        k = k.astype(np.int64) + offset
        offset = int(k.max()) + 1
        a_parts.append(a); y_parts.append(y); k_parts.append(k); v_parts.append(v)
    A = np.concatenate(a_parts, 0)
    Y = np.concatenate(y_parts, 0)
    K = np.concatenate(k_parts, 0)
    V = np.concatenate(v_parts, 0)
    return A[V], Y[V].astype(np.int64), K[V], data_dirs


def _recall_at_fpr(y_true, scores, pos_label, target_fpr=0.01):
    y_arr = np.asarray(y_true)
    if (y_arr == pos_label).sum() == 0 or (y_arr != pos_label).sum() == 0:
        return None
    fpr, tpr, _ = roc_curve(y_arr, scores, pos_label=pos_label)
    mask = fpr <= target_fpr
    if not mask.any():
        return 0.0
    return float(tpr[mask].max())


def _metrics(y_true, y_pred, proba, classes):
    out = {
        "n": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=classes),
    }
    if len(classes) == 2:
        try:
            out["auc"] = roc_auc_score(y_true, proba[:, 1])
        except ValueError:
            out["auc"] = None
        out["recall_at_1pct_fpr"] = _recall_at_fpr(
            y_true, proba[:, 1], pos_label=classes[1], target_fpr=0.01,
        )
    return out


def _conv_aggregate(proba, y_true, conv_id, classes):
    uniq = np.unique(conv_id)
    P, Yc = [], []
    for c in uniq:
        m = conv_id == c
        P.append(proba[m].mean(0)); Yc.append(int(y_true[m][0]))
    P = np.stack(P); Yc = np.array(Yc)
    return Yc, classes[P.argmax(1)], P


# ── Train (with cache) ──────────────────────────────────────────────────────

def train_probe(*, cache_dir=None, force_retrain=False, **cfg):
    """Train one probe — caching enabled by default.

    Caching:
      - Cache key = sha1(canonical cfg)[:12].
      - On a hit, the saved probe.pkl is loaded; no data is touched.
      - Pass `force_retrain=True` to overwrite the cache for this cfg.

    Outputs saved under {cache_dir or CACHE_DIR}/<hash>/:
      - config.json (human-readable)
      - probe.pkl   (cfg + scaler + clf + holdout + split bookkeeping)
    """
    unknown = set(cfg) - _VALID_KEYS
    if unknown:
        raise TypeError(
            f"train_probe got unexpected kwarg(s) {sorted(unknown)}. "
            f"Valid keys are {sorted(_DEFAULTS)} (all lowercase)."
        )
    c = {**_DEFAULTS, **cfg}
    cfg_hash = _cfg_hash(c)
    paths = _paths(cfg_hash, cache_dir)
    verbose = cfg.get("verbose", True)

    if not force_retrain and os.path.exists(paths["probe"]):
        with open(paths["probe"], "rb") as f:
            probe = pickle.load(f)
        if verbose:
            print(f"[cache hit] probe {cfg_hash}  (loaded from {paths['probe']})")
        probe["cfg_hash"] = cfg_hash
        probe["cache_dir"] = cache_dir or CACHE_DIR
        return probe

    A, Y, K, data_dirs = _load_pooled(
        c["model"], c["layer"], c["datasets"], c["judge"],
        c["cat_to_label"], c["jmin"], c["jmax"], verbose=verbose,
    )
    if len(Y) == 0:
        raise RuntimeError(f"No valid tokens after loading {data_dirs}")

    per_conv_label = {int(k): int(Y[K == k][0]) for k in np.unique(K)}
    convs = np.array(sorted(per_conv_label))
    cls_lbls = np.array([per_conv_label[int(kk)] for kk in convs])
    stratify = cls_lbls if all(n >= 2 for n in Counter(cls_lbls.tolist()).values()) else None
    tr_convs, te_convs = train_test_split(
        convs, test_size=c["test_fraction"], random_state=c["random_seed"],
        shuffle=True, stratify=stratify,
    )
    tr_set, te_set = set(tr_convs.tolist()), set(te_convs.tolist())
    is_tr = np.array([int(kk) in tr_set for kk in K])
    is_te = ~is_tr

    def _maybe_mean(mask):
        if c["probe_kind"] == "lr":
            return A[mask], Y[mask], K[mask]
        K_s = K[mask]; A_s = A[mask]; Y_s = Y[mask]
        order = np.argsort(K_s, kind="stable")
        K_s = K_s[order]; A_s = A_s[order]; Y_s = Y_s[order]
        uniq, starts = np.unique(K_s, return_index=True)
        ends = np.r_[starts[1:], len(K_s)]
        X = np.stack([A_s[s:e].mean(0) for s, e in zip(starts, ends)])
        y = np.array([Y_s[s] for s in starts])
        return X, y, uniq

    Xtr_raw, ytr, _   = _maybe_mean(is_tr)
    Xte_raw, yte, Kte = _maybe_mean(is_te)

    scaler_ = StandardScaler() if c["normalize"] else None
    Xtr = scaler_.fit_transform(Xtr_raw) if scaler_ is not None else Xtr_raw
    Xte = scaler_.transform(Xte_raw)      if scaler_ is not None else Xte_raw

    clf_ = LogisticRegression(
        C=1.0 / c["reg_coeff"], fit_intercept=c["fit_intercept"],
        random_state=c["random_seed"], max_iter=2000, solver="lbfgs",
    )
    clf_.fit(Xtr, ytr)

    pred_te = clf_.predict(Xte); prob_te = clf_.predict_proba(Xte)
    holdout = {"unit": "mlr" if c["probe_kind"] == "mlr" else "lr_token",
               **_metrics(yte, pred_te, prob_te, clf_.classes_)}
    if c["probe_kind"] == "lr":
        Yc, predc, Pc = _conv_aggregate(prob_te, yte, Kte, clf_.classes_)
        holdout["conv"] = _metrics(Yc, predc, Pc, clf_.classes_)

    probe = {
        "cfg": c, "cfg_hash": cfg_hash, "cache_dir": cache_dir or CACHE_DIR,
        "scaler": scaler_, "clf": clf_, "classes": clf_.classes_,
        "train_convs": tr_set, "test_convs": te_set, "data_dirs": data_dirs,
        "holdout": holdout,
    }

    # Persist
    os.makedirs(paths["dir"], exist_ok=True)
    with open(paths["probe"], "wb") as f:
        pickle.dump(probe, f)
    with open(paths["config"], "w") as f:
        json.dump({
            "cfg_hash": cfg_hash,
            "cfg":      _canon_cfg(c),
            "data_dirs": data_dirs,
            "n_train_convs": len(tr_set),
            "n_test_convs":  len(te_set),
            "classes":  clf_.classes_.tolist(),
        }, f, indent=2)
    if verbose:
        print(f"[saved] probe {cfg_hash} -> {paths['dir']}/")
    return probe


# ── Evaluate (with cache) ───────────────────────────────────────────────────

def evaluate_probe(probe, datasets, *, judge=None, cat_to_label=None,
                   jmin=None, jmax=None, force_reeval=False, verbose=True):
    """Apply a trained probe to other datasets.

    Defaults pulled from probe['cfg']. Results are cached per (probe, eval-args)
    in {cache_dir}/<probe_hash>/results.pkl. Pass `force_reeval=True` to overwrite.
    """
    pc = probe["cfg"]
    eval_cfg = {
        "datasets":     datasets,
        "judge":        judge        if judge        is not None else pc["judge"],
        "cat_to_label": cat_to_label if cat_to_label is not None else pc["cat_to_label"],
        "jmin":         jmin         if jmin         is not None else pc["jmin"],
        "jmax":         jmax         if jmax         is not None else pc["jmax"],
    }
    eval_key = _eval_key(eval_cfg)
    cfg_hash = probe.get("cfg_hash") or _cfg_hash(pc)
    cache_dir = probe.get("cache_dir") or CACHE_DIR
    results = _load_results(cfg_hash, cache_dir)

    if not force_reeval and eval_key in results:
        if verbose:
            print(f"  [cache hit] eval {eval_key}  ({datasets})")
            cv = results[eval_key].get("conv")
            if cv is not None and cv.get("auc") is not None:
                print(f"    conv AUC={cv['auc']:.3f}"
                      + (f"  rec@1%fpr={cv['recall_at_1pct_fpr']:.3f}"
                         if cv.get('recall_at_1pct_fpr') is not None else ""))
        return results[eval_key]

    A, Y, K, data_dirs = _load_pooled(
        pc["model"], pc["layer"], eval_cfg["datasets"],
        eval_cfg["judge"], eval_cfg["cat_to_label"],
        eval_cfg["jmin"], eval_cfg["jmax"], verbose=verbose,
    )
    if len(Y) == 0:
        return None
    X = probe["scaler"].transform(A) if probe["scaler"] is not None else A

    if pc["probe_kind"] == "mlr":
        order = np.argsort(K, kind="stable")
        K = K[order]; X = X[order]; Y = Y[order]
        uniq, starts = np.unique(K, return_index=True)
        ends = np.r_[starts[1:], len(K)]
        X = np.stack([X[s:e].mean(0) for s, e in zip(starts, ends)])
        Y = np.array([Y[s] for s in starts])
        K = uniq

    pred  = probe["clf"].predict(X)
    proba = probe["clf"].predict_proba(X)
    classes = probe["classes"]
    out = _metrics(Y, pred, proba, classes)
    out["eval_cfg"] = _eval_canon(eval_cfg)
    out["data_dirs"] = data_dirs
    if pc["probe_kind"] == "lr":
        Yc, predc, Pc = _conv_aggregate(proba, Y, K, classes)
        out["conv"] = _metrics(Yc, predc, Pc, classes)

    results[eval_key] = out
    _save_results(cfg_hash, results, cache_dir)
    if verbose:
        line = (f"  acc={out['accuracy']:.3f}  bal={out['balanced_accuracy']:.3f}"
                + (f"  AUC={out['auc']:.3f}" if out.get('auc') is not None else "")
                + f"  (n={out['n']})")
        if "conv" in out:
            cv = out["conv"]
            line += (f"   |  conv: acc={cv['accuracy']:.3f}  bal={cv['balanced_accuracy']:.3f}"
                     + (f"  AUC={cv['auc']:.3f}" if cv.get('auc') is not None else "")
                     + (f"  rec@1%fpr={cv['recall_at_1pct_fpr']:.3f}"
                        if cv.get('recall_at_1pct_fpr') is not None else "")
                     + f"  (n={cv['n']})")
        print(f"  [eval saved] {eval_key}  ({datasets})")
        print(line)
    return out


# ── Grid + sweep glue ───────────────────────────────────────────────────────

def grid_configs(**axes):
    unknown = set(axes) - _VALID_KEYS
    if unknown:
        raise TypeError(
            f"grid_configs got unknown axis/axes {sorted(unknown)}. "
            f"Valid keys are {sorted(_DEFAULTS)} (all lowercase)."
        )
    keys = list(axes.keys())
    vals = [v if isinstance(v, (list, tuple)) else [v] for v in axes.values()]
    return [dict(zip(keys, combo)) for combo in itertools.product(*vals)]


def _cfg_label(cfg):
    diffs = []
    for k, v in cfg.items():
        if k in _DEFAULTS and v != _DEFAULTS[k]:
            diffs.append(f"{k}={v}")
    return ",".join(diffs) if diffs else "(default)"


def run_experiments(train_cfgs, eval_on=None, *, cache_dir=None,
                    force_retrain=False, force_reeval=False, verbose=True):
    """Train + evaluate each config in train_cfgs. Caches every step.

    Caching:
      - `force_retrain=True` retrains every probe (overwriting probe.pkl).
      - `force_reeval=True`  recomputes every eval result (overwriting results.pkl entries).
      - `cache_dir` defaults to CACHE_DIR (= autocomp2/probes/).
    """
    eval_on = eval_on or []
    norm_eval = []
    for e in eval_on:
        if isinstance(e, str):
            norm_eval.append({"name": e, "datasets": e})
        elif isinstance(e, dict):
            e = dict(e)
            e.setdefault("name", str(e.get("datasets", "?")))
            norm_eval.append(e)
        else:
            raise TypeError(f"eval_on entry must be str or dict, got {type(e)}")

    results = []
    for i, cfg in enumerate(train_cfgs):
        label = _cfg_label(cfg)
        if verbose:
            print(f"\n[{i+1}/{len(train_cfgs)}] train: {label}")
        probe = train_probe(cache_dir=cache_dir, force_retrain=force_retrain,
                            verbose=False, **cfg)
        if verbose:
            ho = probe["holdout"]
            line = (f"  holdout  acc={ho['accuracy']:.3f}  bal={ho['balanced_accuracy']:.3f}"
                    + (f"  AUC={ho['auc']:.3f}" if ho.get('auc') is not None else "")
                    + f"  (n={ho['n']})")
            if "conv" in ho:
                cv = ho["conv"]
                line += (f"   |  conv: acc={cv['accuracy']:.3f}  bal={cv['balanced_accuracy']:.3f}"
                         + (f"  AUC={cv['auc']:.3f}" if cv.get('auc') is not None else "")
                         + (f"  rec@1%fpr={cv['recall_at_1pct_fpr']:.3f}"
                            if cv.get('recall_at_1pct_fpr') is not None else "")
                         + f"  (n={cv['n']})")
            print(f"  hash={probe['cfg_hash']}")
            print(line)
        transfers = {}
        for e in norm_eval:
            kw = {k: v for k, v in e.items() if k not in ("name", "datasets")}
            if verbose:
                print(f"  eval  {e['name']}:")
            transfers[e["name"]] = evaluate_probe(
                probe, e["datasets"], verbose=verbose,
                force_reeval=force_reeval, **kw,
            )
        results.append({"label": label, "cfg": probe["cfg"],
                        "probe": probe, "transfers": transfers})
    return results


# ── Summary table ───────────────────────────────────────────────────────────

def _fmt_model(v):     return v.split("/", 1)[-1] if isinstance(v, str) else str(v)
def _fmt_datasets(v):  return v if isinstance(v, str) else "+".join(v)
def _fmt_dict(v):      return "{" + ",".join(f"{k}:{vv}" for k, vv in sorted(v.items())) + "}"
def _fmt_bool(v):      return "T" if v else "F"
def _fmt_num(v):       return f"{v:g}" if isinstance(v, float) else str(v)

_PARAM_COLS = [
    ("model",       lambda c: _fmt_model(c["model"])),
    ("layer",       lambda c: _fmt_num(c["layer"])),
    ("datasets",    lambda c: _fmt_datasets(c["datasets"])),
    ("judge",       lambda c: c["judge"]),
    ("cat→lbl",     lambda c: _fmt_dict(c["cat_to_label"])),
    ("jmin",        lambda c: _fmt_num(c["jmin"])),
    ("jmax",        lambda c: _fmt_num(c["jmax"])),
    ("kind",        lambda c: c["probe_kind"]),
    ("reg_coeff",   lambda c: _fmt_num(c["reg_coeff"])),
    ("norm",        lambda c: _fmt_bool(c["normalize"])),
    ("fit_int",     lambda c: _fmt_bool(c["fit_intercept"])),
    ("test_frac",   lambda c: _fmt_num(c["test_fraction"])),
    ("seed",        lambda c: _fmt_num(c["random_seed"])),
]


def summarize_results(results):
    """Conv-level summary: hyperparameters + holdout + eval-set columns."""
    if not results:
        print("(no results)")
        return
    eval_names = list(results[0]["transfers"].keys())
    result_cols = ["holdout"] + eval_names

    def _cell(m):
        if m is None or "conv" not in m:
            return ""
        cv = m["conv"]
        auc = cv.get("auc")
        if auc is None:
            return ""
        rec = cv.get("recall_at_1pct_fpr")
        rec_str = f"{rec:.3f}" if rec is not None else "  -- "
        return f"{auc:.3f} ({rec_str})"

    param_headers = [h for h, _ in _PARAM_COLS] + ["hash"]
    param_rows    = [
        [fn(r["probe"]["cfg"]) for _, fn in _PARAM_COLS] + [r["probe"]["cfg_hash"]]
        for r in results
    ]
    result_rows   = [[_cell(r["probe"]["holdout"])]
                     + [_cell(r["transfers"][c]) for c in eval_names]
                     for r in results]
    result_w_min  = max(16, max(len(c) for c in result_cols) + 2)

    n_params, n_results = len(param_headers), len(result_cols)
    widths = [None] * (n_params + n_results)
    for j in range(n_params):
        widths[j] = max(len(param_headers[j]),
                        max(len(row[j]) for row in param_rows))
    for j in range(n_results):
        widths[n_params + j] = max(result_w_min, len(result_cols[j]),
                                   max(len(row[j]) for row in result_rows))

    def _row(cells, sep="  "):
        return sep.join(f"{cells[i]:>{widths[i]}}" for i in range(len(cells)))

    print(_row(param_headers + result_cols))
    print("-" * (sum(widths) + 2 * (len(widths) - 1)))
    for prow, rrow in zip(param_rows, result_rows):
        print(_row(prow + rrow))
    print("\n(conversation-level only; cells = `AUROC (recall@1%FPR)`; "
          "blank = no conv-level binary AUROC. Probes cached in "
          f"{CACHE_DIR}/<hash>/.)")

In [28]:
# Example sweep: scan layers x reg_coeff x judges, evaluate each probe on the same two datasets.
sweep_cfgs = grid_configs(
    layer=[12, 24],
    judge=["gpt-5.4-nano"],          # add "gpt-4o-mini" to sweep judges too
    datasets="more-roleplay",
    reg_coeff=[1.0, 30.0],     # all kwargs lowercase — typos raise TypeError
    jmax=[10],
)

results = run_experiments(
    sweep_cfgs,
    eval_on=["more-roleplay", "autoconv10"],
    verbose=False,
)

print()
summarize_results(results)
# Inspect one probe: results[3]["probe"]["clf"], results[3]["transfers"]["autoconv10"]["conv"]


                model  layer       datasets         judge      cat→lbl  jmin  jmax  kind  reg_coeff  norm  fit_int  test_frac  seed          hash           holdout     more-roleplay        autoconv10
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Llama-3.1-8B-Instruct     12  more-roleplay  gpt-5.4-nano  {-2:-1,2:1}     0    10    lr          1     T        F        0.1    42  8d0e1005f1f3     0.751 (0.281)     0.997 (0.936)     0.872 (0.053)
Llama-3.1-8B-Instruct     12  more-roleplay  gpt-5.4-nano  {-2:-1,2:1}     0    10    lr         30     T        F        0.1    42  13be8305c9a7     0.779 (0.094)     0.989 (0.822)     0.881 (0.053)
Llama-3.1-8B-Instruct     24  more-roleplay  gpt-5.4-nano  {-2:-1,2:1}     0    10    lr          1     T        F        0.1    42  21d180318778     0.754 (0.000)     0.991 (0.946)     0.854 (0.158)

In [29]:
# Example sweep: scan layers x reg_coeff x judges, evaluate each probe on the same two datasets.
sweep_cfgs = grid_configs(
    layer=[10, 15, 21, 31],
    judge=["gpt-5.4-nano"],          # add "gpt-4o-mini" to sweep judges too
    datasets="more-roleplay",
    reg_coeff=[1.0, 30.0],     # all kwargs lowercase — typos raise TypeError
    jmax=[10],
    model='google/gemma-2-9b-it'
)

results = run_experiments(
    sweep_cfgs,
    eval_on=["more-roleplay", "autoconv10"],
    verbose=False,
)

print()
summarize_results(results)
# Inspect one probe: results[3]["probe"]["clf"], results[3]["transfers"]["autoconv10"]["conv"]


        model  layer       datasets         judge      cat→lbl  jmin  jmax  kind  reg_coeff  norm  fit_int  test_frac  seed           holdout     more-roleplay        autoconv10
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
gemma-2-9b-it     10  more-roleplay  gpt-5.4-nano  {-2:-1,2:1}     0    10    lr          1     T        F        0.1    42     0.687 (0.050)     0.994 (0.932)     0.662 (0.156)
gemma-2-9b-it     10  more-roleplay  gpt-5.4-nano  {-2:-1,2:1}     0    10    lr         30     T        F        0.1    42     0.715 (0.125)     0.987 (0.815)     0.715 (0.250)
gemma-2-9b-it     15  more-roleplay  gpt-5.4-nano  {-2:-1,2:1}     0    10    lr          1     T        F        0.1    42     0.756 (0.050)     0.997 (0.937)     0.759 (0.125)
gemma-2-9b-it     15  more-roleplay  gpt-5.4-nano  {-2:-1,2:1}     0    10    lr         30     T        F   

In [29]:
# Example sweep: scan layers x reg_coeff x judges, evaluate each probe on the same two datasets.

layers = {
    "meta-llama/Llama-3.1-8B-Instruct": [8, 12, 16, 24],
    #"meta-llama/Llama-3.3-70B-Instruct": [20, 30, 40, 60],
    "google/gemma-2-9b-it": [10, 15, 21, 31],
    "google/gemma-3-12b-it": [12, 18, 24, 36],
    #"Qwen/Qwen2.5-32B-Instruct": [16, 24, 32, 48],
    "Qwen/Qwen2.5-72B-Instruct": [20, 30, 40, 60],
}


for model in layers.keys():
    print(f"{model=}")
    sweep_cfgs = grid_configs(
        model=model,
        layer=layers[model],
        judge=["gpt-5.4-nano"],          # add "gpt-4o-mini" to sweep judges too
        datasets="more-roleplay",
        reg_coeff=[1.0, 10.0, 50.0, 75.0, 100.0],       # all kwargs lowercase — typos raise TypeError
        jmax=[5, 10, 30, 50],
    )
    results = run_experiments(
        sweep_cfgs,
        eval_on=["more-roleplay", "autoconv10"],
        verbose=True,
    )
    
    summarize_results(results)

model='meta-llama/Llama-3.1-8B-Instruct'

[1/80] train: layer=8,jmax=5
  hash=a703a0abb02e
  holdout  acc=0.647  bal=0.625  AUC=0.672  (n=552)   |  conv: acc=0.685  bal=0.671  AUC=0.714  rec@1%fpr=0.250  (n=92)
  eval  more-roleplay:
Loading ./data-more-roleplay/meta-llama--Llama-3.1-8B-Instruct  layer=8 (idx=0)
  token window: j in [0, 5]  (6 positions out of m=100)
  scored file: scored-20260513-022705-meta-llama--Llama-3.1-8B-Instruct.json  (n=1000)
  N=1000  M=6  d=4096  valid_tokens=5,463 / 6,000  convs_kept=911 / 1,000
  [eval saved] 4a2e13cf382a  (more-roleplay)
  acc=0.964  bal=0.962  AUC=0.974  (n=5463)   |  conv: acc=0.968  bal=0.967  AUC=0.989  rec@1%fpr=0.930  (n=911)
  eval  autoconv10:
Loading ./data-autoconv10/meta-llama--Llama-3.1-8B-Instruct  layer=8 (idx=0)
  token window: j in [0, 5]  (6 positions out of m=100)
  scored file: scored-20260421-082038-meta-llama--Llama-3.1-8B-Instruct.json  (n=200)
  N=200  M=6  d=4096  valid_tokens=1,086 / 1,200  convs_kept=181 / 200
 

In [2]:
x = [0,1,2,3]
print(x[0:2], x[2:4])

[0, 1] [2, 3]


In [13]:
# Example usage:
#results = evaluate_on("autoconv10")
# results = evaluate_on(["autoconv10", "more-roleplay"], judge="gpt-4o-mini")
results = evaluate_on("autoconv10", jmin=0, jmax=9)

Loading ./data-autoconv10/meta-llama--Llama-3.1-8B-Instruct  layer=16 (idx=2)
  token window: j in [0, 9]  (10 positions out of m=100)
  scored file: scored-20260421-082038-meta-llama--Llama-3.1-8B-Instruct.json  (n=200)
  N=200  M=10  d=4096  valid_tokens=1,810 / 2,000  convs_kept=181 / 200
--- autoconv10  (1,810 tokens) ---
  accuracy         = 0.7403
  balanced accuracy = 0.7275
  AUC              = 0.8056
    p= -1  p=  1
    t= -1   1072    358
    t=  1    112    268
  conv-level:  acc=0.7956  bal_acc=0.7933  AUC=0.8528  (n=181)
